# 시간 정보를 쓰면 원판 중심 궤적의 공백을 줄일 수 있을까?

준비~받기 원본 77프레임에서 **프레임별 YOLO와 직전 궤적을 이용한 저신뢰 후보 연결**을 비교합니다. 같은 베이스라인의 원본 영상 후보를 사용합니다. 새로운 학습, 프레임 보간, 세 번째 추적기는 사용하지 않습니다. a03 30·31번의 복구 여부와 a05의 기존 좌표 보존 여부를 확인합니다.

In [ ]:
from contextlib import closing
from itertools import islice

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display

from wlcv.config import ANNOTATIONS, RUNS, VIDEOS
from wlcv.data.frames import frames_rgb, read_rgb
from wlcv.evaluation.plate_track import KEY, select_centers, score_centers, summarize_centers

report = RUNS / "plate_baseline/report"
out = report / "temporal_center_tracking"
out.mkdir(exist_ok=True)
source = report / "temporal_misses"  # 06번 노트북의 best.pt 원본 영상 추론 결과
boxes = pd.read_csv(source / "per_frame.csv").query("threshold == 0.10").sort_values(KEY)
centers = pd.read_csv(ANNOTATIONS / "temporal_plate_centers.csv")
assert len(centers) == len(boxes) == 77
assert not centers.duplicated(KEY).any() and not boxes.duplicated(KEY).any()
assert centers[["manual_x", "manual_y"]].notna().eq(centers.visibility.ne("unreadable"), axis=0).all().all()
display(centers.groupby(["video_id", "visibility"]).size().unstack(fill_value=0))

## 1. 수동 중심 확인

검수용 박스로 원본 영상을 확대하고, 모델 후보를 표시하지 않은 상태에서 프레임마다 가까운 원판 면의 동심원 허브 중심(축 결합부)을 육안으로 읽어 원본 픽셀에 표시했습니다. 렌즈 쪽으로 튀어나온 축 끝은 중심에서 벗어나 보일 수 있습니다. 중심부가 가려지면 좌표를 비웠고, 흐림·부분 가림이 있지만 허브를 읽을 수 있으면 `approximate`로 구분했습니다. 수동 좌표는 1회 시각 판독이며 보간하지 않았습니다.

In [ ]:
# 그림에 쓰는 정답 박스는 확대 범위만 정합니다.
review = centers.merge(boxes[KEY + ["x1", "y1", "x2", "y2"]].rename(
    columns={c:"gt_"+c for c in ["x1", "y1", "x2", "y2"]}), on=KEY, validate="one_to_one")

In [ ]:
def draw_review(ax, rgb, row, prediction=False):
    width = row.gt_x2 - row.gt_x1
    left = max(0, int(row.gt_x1 - .26*width))
    right = min(rgb.shape[1], int(row.gt_x2 + .26*width) + 1)
    top = max(0, int(row.gt_y1 - .20*width))
    bottom = min(rgb.shape[0], int(row.gt_y2 + .35*width) + 1)
    ax.imshow(rgb[top:bottom, left:right], extent=(left-.5, right-.5, bottom-.5, top-.5),
              interpolation="nearest")
    ax.set(xticks=[], yticks=[])
    if pd.notna(row.manual_x):
        ax.plot(row.manual_x, row.manual_y, marker="+", ms=12, mew=1.8, color="#ffdc57")
    if prediction and pd.notna(row.cx):
        ax.add_patch(Rectangle((row.x1, row.y1), row.x2-row.x1, row.y2-row.y1,
                               fill=False, lw=1.2, ec="#25d5ed"))
        ax.plot(row.cx, row.cy, marker="o", ms=8, mfc="none", mec="#25d5ed", mew=1.4)


def plot_review_pages(review):
    paths = []
    for video, group in review.groupby("video_id"):
        group = group.set_index("frame_index")
        fig, axes = plt.subplots(5, 8, figsize=(14, 9.5), layout="constrained")
        with closing(frames_rgb(VIDEOS / f"{video}.mp4")) as video_frames:
            for idx, rgb in enumerate(islice(video_frames, group.index.max()+1)):
                if idx not in group.index:
                    continue
                ax = axes.flat[group.index.get_loc(idx)]
                row = group.loc[idx]
                draw_review(ax, rgb, row)
                status = {"visible":"", "approximate":" ~", "unreadable":" ?"}[row.visibility]
                ax.set_title(f"{idx}{status}", fontsize=10)
        for ax in list(axes.flat)[len(group):]:
            ax.set_visible(False)
        fig.suptitle(f"{video}  /  Manual hub centers     + point     ~ approximate     ? unreadable")
        path = out / f"{video}_centers.jpg"
        fig.savefig(path, dpi=125, facecolor="white")
        plt.close(fig)
        paths.append(path)
    return paths

In [ ]:
for path in plot_review_pages(review):
    display(Image.open(path))

## 2. 원판 후보 선택

각 영상의 시작에서 이력을 초기화합니다. 두 방법 모두 0.10 이상이면 해당 프레임의 최고 신뢰도 후보를 사용합니다. 연결 방법은 고신뢰 관측이 있던 시점부터 최대 두 프레임 동안만 후보 하한 0.001까지 봅니다. 마지막 두 관측의 이동량으로 현재 위치를 예상하고, 마지막 박스 너비 절반 이내에서 가장 가까운 후보를 고릅니다. 과거 점이 하나면 그 위치를 기준으로 합니다. 연결할 후보가 없으면 결측입니다. 수동 중심과 정답 IoU는 선택 뒤 평가에만 사용합니다.

In [ ]:
candidates = pd.read_csv(source / "candidates.csv")
selected = select_centers(boxes[KEY], candidates[KEY + ["confidence", "x1", "y1", "x2", "y2"]])
result = score_centers(selected, centers, boxes)
summary = summarize_centers(result)
result.to_csv(out / "per_frame.csv", index=False)
summary.to_csv(out / "summary.csv", index=False)
display(summary.round(3))
display(result[result.source == "linked"][KEY + ["confidence", "iou", "error_px", "error_width_pct"]].round(3))

## 3. 중심이 이어지는 위치와 복구한 후보 확인

x·y는 원본 영상 좌표입니다. 검출을 못 했으면 선을 끊습니다. 정탐은 기존 정답 박스 IoU ≥ 0.50으로 판정합니다. 중심 오차는 판독 가능하고 정탐인 프레임에서 수동 중심과 선택된 박스 중심 사이의 유클리드 거리이며, 정답 박스 너비 대비 비율도 함께 기록합니다. 초기 공백과 최고점 오차는 따로 봅니다. 최고점은 판독 가능한 준비~받기 구간에서 관측한 최소 y의 첫 프레임입니다.

In [ ]:
def plot_tracks(detail):
    fig, axes = plt.subplots(2, 2, figsize=(11, 6.6), layout="constrained")
    for row, (video, g) in enumerate(detail.groupby("video_id")):
        yolo = g[g.method == "YOLO"].sort_values("frame_index")
        past = g[g.method == "Past link"].sort_values("frame_index")
        for ax, axis in zip(axes[row], ["x", "y"]):
            ax.plot(yolo.frame_index, yolo["manual_" + axis], color="#172b38", lw=1.3,
                    marker=".", ms=5, label="Manual hub")
            ax.plot(past.frame_index, past["c" + axis], "--", color="#d4782c", lw=1.8,
                    label="Past link")
            ax.scatter(yolo.frame_index, yolo["c" + axis], marker="o", s=19,
                       facecolors="none", edgecolors="#398a99", label="Frame YOLO", zorder=3)
            new = past[past.source == "linked"]
            ax.scatter(new.frame_index, new["c" + axis], marker="D", color="#cc5f16",
                       s=36, label="Linked low score", zorder=4)
            ax.set(title=f"{video[:3]}  /  {axis} coordinate", xlabel="Video frame",
                   ylabel=f"Image {axis} (px)")
            ax.set_xlim(yolo.frame_index.min() - .5, yolo.frame_index.max() + .5)
            valid = past[past.cx.notna()]
            if len(valid) and valid.frame_index.min() > yolo.frame_index.min():
                ax.axvspan(yolo.frame_index.min()-.5, valid.frame_index.min()-.5,
                           color="#b6bdc4", alpha=.22, zorder=0)
                ax.text(.02, .94, "No active track", transform=ax.transAxes,
                        ha="left", va="top", fontsize=8, color="#47515b")
            ax.grid(axis="y", alpha=.18)
            if axis == "y":
                ax.invert_yaxis()
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper center", ncols=4, frameon=False)
    return fig

In [ ]:
def plot_examples(detail):
    past = detail[detail.method == "Past link"].set_index(["video_id", "frame_index"])
    panels = [("a03_sn_f_01", 2), ("a03_sn_f_01", 30),
              ("a03_sn_f_01", 31), ("a05_sn_f_01", 31)]
    fig, axes = plt.subplots(1, 4, figsize=(11, 3.6), layout="constrained")
    for ax, (video, idx) in zip(axes, panels):
        row = past.loc[video, idx]
        rgb = read_rgb(VIDEOS / f"{video}.mp4", idx)
        draw_review(ax, rgb, row, prediction=True)
        ax.set_title(f"{video[:3]}  /  frame {idx}", fontsize=12)
        message = {"linked":"Low-score link", "high":"Both methods", "missing":"No output"}[row.source]
        if pd.notna(row.error_px):
            message += f"  /  {row.error_px:.1f} px"
        ax.set_xlabel(message, fontsize=10)
    fig.suptitle("Manual hub (+) and chosen box center (circle)")
    return fig

In [ ]:
for fig, name in [(plot_tracks(result), "center_tracks.png"), (plot_examples(result), "checked_frames.png")]:
    fig.savefig(out / name, dpi=150, facecolor="white")
plt.show()